# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² Mental Health Survey dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
This dataset is described with a [Croissant schema](https://mlcommons.org/croissant/) available at:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as a single object!)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Dataset Published on: {metadata.datePublished}")
print(f"Dataset Version: {metadata.version}")
print(f"Dataset License: {metadata.license}")


## 2. Data Overview
Review the available record sets, fields, and their `@id` references.

This section lists all record sets and their structure using their unique `@id` fields.

In [ ]:
# List all record sets in the dataset with their @id
record_sets = [rs for rs in dataset.record_sets]
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}, Description: {getattr(rs, 'description', None)}")
    # List fields and columns by @id
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}), type: {getattr(field, 'dataType', None)}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id}), type: {getattr(col, 'dataType', None)}")
    print()
if not record_sets:
    print("No record sets found in schema. Please verify dataset structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below we:
- Select a record set by its `@id`,
- Extract all records,
- Build a DataFrame for further exploration.

_Note_: All entities (record sets, fields, columns) use their `@id` for referencing.

In [ ]:
# Collect all record sets (by @id)
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids:", record_set_ids)

# We'll use the first record set as example if available
main_record_set_id = record_set_ids[0] if record_set_ids else None
dataframes = {}
for rs_id in record_set_ids:
    # Each record is a dict with keys as field @ids
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for record set {rs_id}, columns:", df.columns.tolist())

# Show preview
if main_record_set_id:
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set to display.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, and group by key attributes.

We select a numeric field and a group field using their `@id` for demonstration.

In [ ]:
# Choose a record set to work with
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Identify numeric and categorical fields by inspecting columns
print("Columns available for EDA:", df.columns.tolist())

# Example: Assume 'cr:GAD7_total_score' and 'cr:age_years' exist as numeric fields (replace with actual @ids in your dataset)
# Example group field: 'cr:village' or 'cr:gender'
numeric_field_id = None
group_field_id = None

# Try to auto-select typical fields
for field_id in df.columns:
    if 'GAD7' in field_id and 'score' in field_id:
        numeric_field_id = field_id
    elif 'PHQ9' in field_id and 'score' in field_id:
        numeric_field_id = field_id
    elif ('age' in field_id or 'years' in field_id):
        if numeric_field_id is None:
            numeric_field_id = field_id
    if ('village' in field_id or 'gender' in field_id):
        group_field_id = field_id

if numeric_field_id is None:
    # Default to first numeric-looking column
    for c in df.columns:
        if df[c].dtype in [int, float]:
            numeric_field_id = c
            break
if group_field_id is None:
    # Default to first non-numeric column
    for c in df.columns:
        if df[c].dtype == object:
            group_field_id = c
            break

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Filtering
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for filtering and analysis.")

## 5. Visualization
Visualize distributions or relationships using the extracted DataFrame.

Below we plot the distribution of the selected numeric field and compare groups if available.

In [ ]:
# Simple visualization of the numeric field distribution
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].dropna().hist(bins=20, edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Visualize comparison of means by group
if group_field_id and numeric_field_id and group_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
    group_means.plot(kind='bar', figsize=(10, 5))
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset using Croissant schema and the `mlcroissant` library. Key actions included:
- Listing record sets and fields via their `@id`,
- Extracting survey records and visualizing the distribution of mental health scores,
- Filtering and grouping by demographic fields for deeper analysis.

For deeper insights, you can extend this notebook with further visualizations, cross-tabulations, and predictive modeling using field `@id`s as references.